In [4]:
import os, sys

parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))

if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

In [ ]:
from scrapxd.fetcher import Fetcher

f = Fetcher()

csv_path = os.path.join(parent_dir, 'data', 'usernames.csv')
idx = 1

for i in range(1, 68):
    soup = f.fetch_soup(f'https://letterboxd.com/members/popular/page/{i}/')
    user_td = soup.find_all('td', class_='table-person')

    usernames = []

    for td in user_td:
        anchor = td.find('a', class_='avatar -a40')
        username = anchor.get('href').strip('/')

        if username:
            usernames.append((idx, username))
            idx += 1

    with open(csv_path, 'a') as file:
        for user in usernames:
            file.write(f'{user[0]};{user[1]}\n')


In [ ]:
import pandas as pd
from scrapxd import Scrapxd


csv_data_path = os.path.join(parent_dir, 'data', 'users_data.csv')

df = pd.read_csv(csv_path, sep=';', names=['idx', 'username'])
users = df['username'].tolist()

for i in range(len(users)):
    if i % 100 == 0:
        client = Scrapxd()

    username = users[i]
    print(f'Fetching data for user {i+1}/{len(users)}: {username}')
    user = client.get_user(username)
    logs = user.logs

    user_data = [(i, username, entry.film.slug, entry.rating) for entry in logs.entries if entry.rating is not None]

    with open(csv_data_path, 'a') as file:
        for log in user_data:
            file.write(f'{log[0]};{log[1]};{log[2]};{log[3]}\n')


In [8]:
import pandas as pd

user_df = pd.read_csv(os.path.join(parent_dir, 'data', 'users_data.csv'), sep=';', names=['user_id', 'username', 'slug', 'rating'])
film_slugs = pd.read_csv(os.path.join(parent_dir, 'data', 'films_data.csv'), sep=';')

print(f"unique users {len(user_df['username'].unique())}")
print(f"unique slugs {len(user_df['slug'].unique())}")
print(f"lines: {len(user_df)}")

filtered_df = user_df[user_df['slug'].isin(film_slugs['slug'])]
print(f"filtered unique users {len(filtered_df['username'].unique())}")
print(f"filtered unique slugs {len(filtered_df['slug'].unique())}")
print(f"filtered lines: {len(filtered_df)}")

unique users 1977
unique slugs 241426
lines: 4882233
filtered unique users 1976
filtered unique slugs 46990
filtered lines: 4300111
